# 01 - Calidad de datos

Proyecto: Prediccion de caida critica de produccion mensual en pozos no convencionales de petroleo y gas.

Objetivo de este notebook: cargar el dataset principal y revisar su calidad antes de crear el target, hacer EDA profundo, construir features o entrenar modelos.

Reglas metodologicas de esta etapa:
- No se crea todavia `caida_critica`.
- No se entrenan modelos.
- No se hace feature engineering avanzado.
- No se unen datasets externos.
- No se usa `capitulo-iv-pozos.csv`.
- No se eliminan columnas sin justificacion.
- Se usan paths relativos desde la carpeta `notebooks/`.

## 1. Importacion de librerias

Se importan librerias basicas para manipulacion de datos, resumen numerico, visualizacion y manejo de paths.

In [7]:
from pathlib import Path
import re
import unicodedata
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

sns.set_theme(style="whitegrid")

## 2. Carga del dataset principal

El path esperado es `../data/raw/produccion-de-pozos-de-gas-y-petroleo-no-convencional.csv`.

En este workspace el archivo raw puede aparecer con un nombre levemente distinto por codificacion de caracteres. Por eso se intenta primero el path esperado y, solo si no existe, se busca un CSV equivalente dentro de `../data/raw/` sin modificar el archivo original.

In [8]:
raw_data_path = Path("../data/raw/produccion-de-pozos-de-gas-y-petroleo-no-convencional.csv")
raw_dir = Path("../data/raw")

if raw_data_path.exists():
    data_path = raw_data_path
else:
    csv_candidates = sorted(raw_dir.glob("*pozos*no-convencional*.csv"))
    if not csv_candidates:
        csv_candidates = sorted(raw_dir.glob("*.csv"))
    if not csv_candidates:
        raise FileNotFoundError(f"No se encontro ningun CSV en {raw_dir.resolve()}")
    data_path = csv_candidates[0]

print(f"Archivo seleccionado: {data_path}")

Archivo seleccionado: ..\data\raw\produccin-de-pozos-de-gas-y-petrleo-no-convencional.csv


In [9]:
df_raw = pd.read_csv(data_path, low_memory=False)

print(f"Filas: {df_raw.shape[0]:,}")
print(f"Columnas: {df_raw.shape[1]:,}")

Filas: 405,996
Columnas: 40


## 3. Inspeccion inicial

Se revisan dimensiones, primeras filas, ultimas filas, nombres de columnas, tipos de datos y cantidad de valores unicos por columna.

In [10]:
df_raw.shape

(405996, 40)

In [11]:
df_raw.head()

,idempresa,anio,mes,idpozo,prod_pet,prod_gas,prod_agua,iny_agua,iny_gas,iny_co2,iny_otro,tef,vida_util,tipoextraccion,tipoestado,tipopozo,observaciones,fechaingreso,rectificado,habilitado,idusuario,empresa,sigla,formprod,profundidad,formacion,idareapermisoconcesion,areapermisoconcesion,idareayacimiento,areayacimiento,cuenca,provincia,coordenadax,coordenaday,tipo_de_recurso,proyecto,clasificacion,subclasificacion,sub_tipo_recurso,fecha_data
0,YSUR,2016,1,135204,0.0000,59.9400,28.3500,0.0000,0.0000,0.0000,0.0000,30.8100,NaN,Plunger Lift,Extracción Efectiva,Gasífero,NaN,2016-02-17 10:50:46.929347,f,t,5,YSUR ENERGÍA ARGENTINA S.R.L.,APA.Nq.ACO-13(d),PREC,"3,500.0000",precuyo,ANC,ANTICLINAL CAMPAMENTO,ACO,ANTICLINAL CAMPAMENTO OESTE,NEUQUINA,Neuquén,-69.7935,-38.9690,NO CONVENCIONAL,GAS PLUS,EXPLOTACION,DESARROLLO,TIGHT,2016-01-31
1,YSUR,2018,1,155584,80.8060,786.9020,0.0000,0.0000,0.0000,0.0000,0.0000,31.0000,NaN,Surgencia Natural,Extracción Efectiva,Gasífero,NaN,2018-02-10 08:37:14.717426,f,t,444,YSUR ENERGÍA ARGENTINA S.R.L.,YSR.RN.EFO-262(d),LAJA,"3,847.0000",lajas,FEO,ESTACION FERNANDEZ ORO,Z155,ESTACION FERNANDEZ ORO,NEUQUINA,Rio Negro,-67.7979,-39.0287,NO CONVENCIONAL,GAS PLUS,EXPLOTACION,DESARROLLO,TIGHT,2018-01-31
2,YSUR,2015,1,135203,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,NaN,Sin Sistema de Extracción,Parado Transitoriamente,Gasífero,NaN,2015-02-26 13:35:35.533458,f,t,5,YSUR ENERGÍA ARGENTINA S.R.L.,APA.Nq.ACO-12(d),PREC,"3,617.0000",precuyo,ANC,ANTICLINAL CAMPAMENTO,ACO,ANTICLINAL CAMPAMENTO OESTE,NEUQUINA,Neuquén,-69.8055,-38.9628,NO CONVENCIONAL,GAS PLUS,EXPLOTACION,DESARROLLO,TIGHT,2015-01-31
3,YSUR,2017,1,155582,58.2800,608.0100,13.6300,0.0000,0.0000,0.0000,0.0000,31.0000,NaN,Surgencia Natural,Extracción Efectiva,Gasífero,NaN,2017-02-16 13:45:37.233373,f,t,444,YSUR ENERGÍA ARGENTINA S.R.L.,YSR.RN.EFO-245(d),LAJA,"3,805.0000",lajas,FEO,ESTACION FERNANDEZ ORO,Z155,ESTACION FERNANDEZ ORO,NEUQUINA,Rio Negro,-67.8472,-39.0107,NO CONVENCIONAL,GAS PLUS,EXPLOTACION,DESARROLLO,TIGHT,2017-01-31
4,YSUR,2016,1,136137,0.0000,432.1700,45.5200,0.0000,0.0000,0.0000,0.0000,31.0000,NaN,Surgencia Natural,Extracción Efectiva,Gasífero,NaN,2016-02-17 10:50:46.929347,f,t,5,YSUR ENERGÍA ARGENTINA S.R.L.,APA.Nq.Gu-1199(d),PREC,"3,375.0000",precuyo,NDD,AL NORTE DE LA DORSAL,GUA,GUANACO,NEUQUINA,Neuquén,-69.2249,-38.8679,NO CONVENCIONAL,GAS PLUS,EXPLOTACION,DESARROLLO,TIGHT,2016-01-31


In [12]:
df_raw.tail()

,idempresa,anio,mes,idpozo,prod_pet,prod_gas,prod_agua,iny_agua,iny_gas,iny_co2,iny_otro,tef,vida_util,tipoextraccion,tipoestado,tipopozo,observaciones,fechaingreso,rectificado,habilitado,idusuario,empresa,sigla,formprod,profundidad,formacion,idareapermisoconcesion,areapermisoconcesion,idareayacimiento,areayacimiento,cuenca,provincia,coordenadax,coordenaday,tipo_de_recurso,proyecto,clasificacion,subclasificacion,sub_tipo_recurso,fecha_data
405991,ACO,2023,12,164374,282.6445,"2,582.0947",2.0687,0.0000,0.0000,0.0000,0.0000,30.9861,NaN,Surgencia Natural,Extracción Efectiva,Petrolífero,ACO,2024-01-07 23:26:54.919622,f,t,459,Petrolera Aconcagua Energia S.A.,ACO.RN.CB-2003,PROS,"2,840.0000",punta rosada,ELO,ENTRE LOMAS,CHB,CHARCO BAYO,NEUQUINA,Rio Negro,-68.1066,-38.1785,NO CONVENCIONAL,Sin Proyecto,EXPLOTACION,DESARROLLO,TIGHT,2023-12-31
405992,ACO,2023,12,3640,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,NaN,Bombeo Mecánico,Parado Transitoriamente,Petrolífero,ACO,2024-01-07 23:26:54.919622,f,t,459,Petrolera Aconcagua Energia S.A.,PPC.Nq.EC-4,VMUT,"2,585.0000",vaca muerta,ELO,ENTRE LOMAS,ECL,EL CARACOL,NEUQUINA,Neuquén,-68.4524,-37.9542,NO CONVENCIONAL,Sin Proyecto,EXPLOTACION,AVANZADA,SHALE,2023-12-31
405993,ACO,2023,12,146101,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,NaN,Bombeo Mecánico,Parado Transitoriamente,Gasífero,ACO,2024-01-07 23:26:54.919622,f,t,459,Petrolera Aconcagua Energia S.A.,PEL.RN.JQ.e-2,TORD,"3,780.0000",tordillo,AAMA,JARILLA QUEMADA,JQUE,JARILLA QUEMADA,NEUQUINA,Rio Negro,-67.9638,-38.4607,NO CONVENCIONAL,GAS PLUS,EXPLORACION,EXTENSION,TIGHT,2023-12-31
405994,ACO,2023,12,164873,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,NaN,NaN,NaN,Cargado automáticamente como [Sin movimientos],2024-01-08 13:09:12.695966,f,t,459,Petrolera Aconcagua Energia S.A.,PPC.RN.CB-170,PROS,"2,400.0000",punta rosada,ELO,ENTRE LOMAS,CHB,CHARCO BAYO,NEUQUINA,Rio Negro,-68.1297,-38.1497,NO CONVENCIONAL,Sin Proyecto,EXPLOTACION,DESARROLLO,TIGHT,2023-12-31
405995,ACO,2023,12,164872,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,NaN,NaN,NaN,Cargado automáticamente como [Sin movimientos],2024-01-08 13:09:12.695966,f,t,459,Petrolera Aconcagua Energia S.A.,PPC.RN.CB-191(p),PROS,"2,848.0000",punta rosada,ELO,ENTRE LOMAS,CHB,CHARCO BAYO,NEUQUINA,Rio Negro,-68.0999,-38.1743,NO CONVENCIONAL,Sin Proyecto,EXPLOTACION,DESARROLLO,TIGHT,2023-12-31


In [13]:
columns_df = pd.DataFrame({"columna": df_raw.columns})
columns_df

,columna
0,idempresa
1,anio
2,mes
3,idpozo
4,prod_pet
5,prod_gas
6,prod_agua
7,iny_agua
8,iny_gas
9,iny_co2


In [14]:
dtypes_df = df_raw.dtypes.astype(str).rename("tipo_dato").to_frame()
dtypes_df

,tipo_dato
idempresa,str
anio,int64
mes,int64
idpozo,int64
prod_pet,float64
prod_gas,float64
prod_agua,float64
iny_agua,float64
iny_gas,float64
iny_co2,float64


In [15]:
unique_values_df = (
    df_raw.nunique(dropna=True)
    .rename("valores_unicos")
    .sort_values(ascending=False)
    .to_frame()
)

unique_values_df

,valores_unicos
prod_gas,185556
prod_pet,139356
prod_agua,128588
tef,19746
fechaingreso,9949
idpozo,4929
sigla,4744
coordenadax,4690
coordenaday,4435
profundidad,2843


## 4. Columnas candidatas por rol analitico

Esta identificacion no transforma el dataset. Sirve para orientar las siguientes etapas del proyecto y detectar columnas relevantes para calidad, EDA, feature engineering temporal y modelado futuro.

In [16]:
candidate_columns = pd.DataFrame(
    [
        ("Identificador de pozo", "idpozo", "Identificador principal para la unidad pozo-mes."),
        ("Identificador/nombre de pozo", "sigla", "Identificador operativo o nombre corto del pozo."),
        ("Fecha / periodo", "fecha_data", "Fecha mensual de referencia del registro."),
        ("Fecha / periodo", "anio", "Anio del registro."),
        ("Fecha / periodo", "mes", "Mes del registro."),
        ("Fecha administrativa", "fechaingreso", "Fecha de ingreso del registro al sistema."),
        ("Produccion de petroleo", "prod_pet", "Volumen mensual de petroleo."),
        ("Produccion de gas", "prod_gas", "Volumen mensual de gas."),
        ("Produccion de agua", "prod_agua", "Volumen mensual de agua."),
        ("Inyeccion", "iny_agua, iny_gas, iny_co2, iny_otro", "Variables operativas de inyeccion."),
        ("Provincia", "provincia", "Ubicacion administrativa."),
        ("Cuenca", "cuenca", "Ubicacion geologica regional."),
        ("Yacimiento", "areayacimiento, idareayacimiento", "Yacimiento o area de yacimiento."),
        ("Concesion", "areapermisoconcesion, idareapermisoconcesion", "Area, permiso o concesion."),
        ("Empresa / operador", "empresa, idempresa", "Empresa informante u operadora."),
        ("Tipo de recurso", "tipo_de_recurso, sub_tipo_recurso, proyecto", "Clasificacion del recurso no convencional."),
        ("Estado / extraccion", "tipoestado, tipoextraccion, tipopozo", "Estado operativo y tipo de pozo."),
        ("Geologia / completacion", "formacion, formprod, profundidad", "Variables descriptivas del pozo."),
        ("Coordenadas", "coordenadax, coordenaday", "Ubicacion espacial."),
        ("Otras descriptivas", "clasificacion, subclasificacion, rectificado, habilitado, tef", "Variables adicionales a revisar."),
    ],
    columns=["rol", "columnas_candidatas", "comentario"],
)

candidate_columns

,rol,columnas_candidatas,comentario
0,Identificador de pozo,idpozo,Identificador principal para la unidad pozo-mes.
1,Identificador/nombre de pozo,sigla,Identificador operativo o nombre corto del pozo.
2,Fecha / periodo,fecha_data,Fecha mensual de referencia del registro.
3,Fecha / periodo,anio,Anio del registro.
4,Fecha / periodo,mes,Mes del registro.
5,Fecha administrativa,fechaingreso,Fecha de ingreso del registro al sistema.
6,Produccion de petroleo,prod_pet,Volumen mensual de petroleo.
7,Produccion de gas,prod_gas,Volumen mensual de gas.
8,Produccion de agua,prod_agua,Volumen mensual de agua.
9,Inyeccion,"iny_agua, iny_gas, iny_co2, iny_otro",Variables operativas de inyeccion.


## 5. Correcciones minimas para una base de trabajo

Se crea `df_base` a partir de `df_raw` con correcciones minimas y justificadas:
- normalizacion de nombres de columnas para evitar problemas por espacios, mayusculas o caracteres especiales;
- conversion de columnas de fecha con `errors='coerce'` para detectar valores no parseables;
- eliminacion de duplicados exactos solo si existen.

No se crea el target y no se modifican variables productivas.

In [17]:
def normalize_column_name(column_name: str) -> str:
    """Normaliza nombres de columnas a snake_case ASCII."""
    normalized = unicodedata.normalize("NFKD", str(column_name))
    ascii_name = normalized.encode("ascii", "ignore").decode("ascii")
    ascii_name = ascii_name.strip().lower()
    ascii_name = re.sub(r"[^0-9a-zA-Z]+", "_", ascii_name)
    return ascii_name.strip("_")


df_base = df_raw.copy()
original_columns = df_base.columns.tolist()
df_base.columns = [normalize_column_name(col) for col in df_base.columns]

column_name_changes = pd.DataFrame(
    {
        "columna_original": original_columns,
        "columna_normalizada": df_base.columns,
    }
)

column_name_changes[column_name_changes["columna_original"] != column_name_changes["columna_normalizada"]]

,columna_original,columna_normalizada


In [18]:
date_columns = [col for col in df_base.columns if "fecha" in col]
date_conversion_rows = []

for col in date_columns:
    non_null_before = int(df_base[col].notna().sum())
    df_base[col] = pd.to_datetime(df_base[col], errors="coerce")
    non_null_after = int(df_base[col].notna().sum())
    date_conversion_rows.append(
        {
            "columna": col,
            "no_nulos_antes": non_null_before,
            "fechas_validas_despues": non_null_after,
            "valores_no_parseables": non_null_before - non_null_after,
            "fecha_min": df_base[col].min(),
            "fecha_max": df_base[col].max(),
        }
    )

date_conversion_summary = pd.DataFrame(date_conversion_rows)
date_conversion_summary

,columna,no_nulos_antes,fechas_validas_despues,valores_no_parseables,fecha_min,fecha_max
0,fechaingreso,405996,405996,0,2006-11-13 13:57:53.058895,2026-05-20 22:24:18.187222
1,fecha_data,405996,405996,0,2006-01-31 00:00:00.000000,2026-04-30 00:00:00.000000


## 6. Validaciones de calidad de datos

Se revisan duplicados exactos, duplicados por pozo-mes, nulos, valores extremos basicos en produccion, distribucion temporal y cantidad de registros por pozo.

### 6.1 Duplicados exactos

In [19]:
exact_duplicates_count = int(df_base.duplicated().sum())

pd.DataFrame(
    {
        "metrica": ["duplicados_exactos"],
        "valor": [exact_duplicates_count],
    }
)

,metrica,valor
0,duplicados_exactos,0


### 6.2 Duplicados por pozo y mes

La unidad esperada es pozo-mes. Por eso se verifica si existe mas de un registro para la misma combinacion `idpozo`, `anio`, `mes`.

In [20]:
pozo_col = "idpozo"
pozo_mes_keys = ["idpozo", "anio", "mes"]

missing_pozo_mes_keys = [col for col in pozo_mes_keys if col not in df_base.columns]

if missing_pozo_mes_keys:
    raise KeyError(f"Faltan columnas para validar pozo-mes: {missing_pozo_mes_keys}")

pozo_mes_duplicate_mask = df_base.duplicated(subset=pozo_mes_keys, keep=False)
pozo_mes_duplicates = df_base.loc[pozo_mes_duplicate_mask].sort_values(pozo_mes_keys)

pozo_mes_duplicate_groups = (
    df_base.groupby(pozo_mes_keys, dropna=False)
    .size()
    .reset_index(name="n_registros")
    .query("n_registros > 1")
    .sort_values("n_registros", ascending=False)
)

pd.DataFrame(
    {
        "metrica": ["filas_en_duplicados_pozo_mes", "grupos_duplicados_pozo_mes"],
        "valor": [len(pozo_mes_duplicates), len(pozo_mes_duplicate_groups)],
    }
)

,metrica,valor
0,filas_en_duplicados_pozo_mes,0
1,grupos_duplicados_pozo_mes,0


In [21]:
pozo_mes_duplicate_groups.head(20)

,idpozo,anio,mes,n_registros


### 6.3 Nulos por columna

In [22]:
null_summary = pd.DataFrame(
    {
        "nulos": df_base.isna().sum(),
        "pct_nulos": df_base.isna().mean() * 100,
        "no_nulos": df_base.notna().sum(),
        "valores_unicos": df_base.nunique(dropna=True),
    }
).sort_values(["pct_nulos", "nulos"], ascending=False)

null_summary

,nulos,pct_nulos,no_nulos,valores_unicos
vida_util,397379,97.8776,8617,466
observaciones,382947,94.3229,23049,215
clasificacion,910,0.2241,405086,3
subclasificacion,910,0.2241,405086,8
tipoextraccion,605,0.1490,405391,11
tipoestado,605,0.1490,405391,16
tipopozo,605,0.1490,405391,6
sub_tipo_recurso,440,0.1084,405556,2
idempresa,0,0.0000,405996,55
anio,0,0.0000,405996,21


In [23]:
high_null_threshold = 50
high_null_columns = null_summary[null_summary["pct_nulos"] >= high_null_threshold]
high_null_columns

,nulos,pct_nulos,no_nulos,valores_unicos
vida_util,397379,97.8776,8617,466
observaciones,382947,94.3229,23049,215


### 6.4 Valores negativos y ceros en produccion

En esta etapa no se corrigen automaticamente. Se reportan para decidir su tratamiento en EDA o limpieza posterior.

In [24]:
production_cols = [col for col in ["prod_pet", "prod_gas", "prod_agua"] if col in df_base.columns]

production_quality_rows = []
for col in production_cols:
    series = pd.to_numeric(df_base[col], errors="coerce")
    production_quality_rows.append(
        {
            "columna": col,
            "nulos": int(series.isna().sum()),
            "negativos": int((series < 0).sum()),
            "ceros": int((series == 0).sum()),
            "pct_ceros": float((series == 0).mean() * 100),
            "min": series.min(),
            "max": series.max(),
        }
    )

production_quality = pd.DataFrame(production_quality_rows).set_index("columna")
production_quality

,nulos,negativos,ceros,pct_ceros,min,max
columna,,,,,,
prod_pet,0,1,144047,35.4799,-0.0010,"26,593.2610"
prod_gas,0,2,83800,20.6406,-12.2670,"29,129.6600"
prod_agua,0,0,122426,30.1545,0.0000,"34,792.9500"


In [25]:
negative_production_mask = pd.Series(False, index=df_base.index)
for col in production_cols:
    negative_production_mask |= pd.to_numeric(df_base[col], errors="coerce") < 0

negative_production_examples = df_base.loc[
    negative_production_mask,
    ["idpozo", "anio", "mes", "fecha_data", "empresa", "sigla"] + production_cols,
]

negative_production_examples.head(20)

,idpozo,anio,mes,fecha_data,empresa,sigla,prod_pet,prod_gas,prod_agua
289972,153227,2020,5,2020-05-31,PLUSPETROL S.A.,PLU.Nq.Ce-1454(d),1.0200,-7.5190,0.0000
290026,153228,2020,5,2020-05-31,PLUSPETROL S.A.,PLU.Nq.Ce-1454(d),1.6700,-12.2670,0.0000
312211,73804,2008,12,2008-12-31,PETROBRAS ARGENTINA S.A.,Q.SC.AEC.x-1,-0.0010,0.0000,0.0000


### 6.5 Registros por mes, anio y pozo

In [26]:
if "fecha_data" in df_base.columns:
    temporal_series = df_base["fecha_data"]
else:
    temporal_series = pd.to_datetime(
        df_base["anio"].astype(str) + "-" + df_base["mes"].astype(str).str.zfill(2) + "-01",
        errors="coerce",
    )

periodo_mes = temporal_series.dt.to_period("M")

temporal_range = pd.DataFrame(
    {
        "fecha_min": [temporal_series.min()],
        "fecha_max": [temporal_series.max()],
        "meses_unicos": [periodo_mes.nunique(dropna=True)],
        "anios_unicos": [df_base["anio"].nunique(dropna=True)],
        "pozos_unicos": [df_base["idpozo"].nunique(dropna=True)],
    }
)

temporal_range

,fecha_min,fecha_max,meses_unicos,anios_unicos,pozos_unicos
0,2006-01-31,2026-04-30,244,21,4929


In [27]:
records_by_month = (
    df_base.groupby(periodo_mes)
    .size()
    .rename("registros")
    .to_frame()
)

records_by_month.head(), records_by_month.tail()

(            registros
 fecha_data           
 2006-01           195
 2006-02           196
 2006-03           197
 2006-04           197
 2006-05           197,
             registros
 fecha_data           
 2025-12          4740
 2026-01          4772
 2026-02          4814
 2026-03          4851
 2026-04          4899)

In [28]:
records_by_year = (
    df_base.groupby("anio")
    .size()
    .rename("registros")
    .to_frame()
)

records_by_year

,registros
anio,
2006,2368
2007,2442
2008,2280
2009,1698
2010,2038
2011,2599
2012,3640
2013,5184
2014,8777


In [29]:
records_by_well = df_base.groupby("idpozo").size().rename("registros")

records_by_well.describe().to_frame()

,registros
count,"4,929.0000"
mean,82.3688
std,57.3861
min,1.0000
25%,31.0000
50%,80.0000
75%,125.0000
max,244.0000


## 7. Verificacion de unidad de analisis

Si `idpozo + anio + mes` es unico, el dataset es compatible con una unidad de analisis pozo-mes. Si aparecen duplicados, deben revisarse antes de modelar y no se resuelven automaticamente en este notebook.

In [30]:
n_rows = len(df_base)
n_unique_pozo_mes = df_base[pozo_mes_keys].drop_duplicates().shape[0]

unit_check = pd.DataFrame(
    {
        "metrica": ["filas", "combinaciones_unicas_idpozo_anio_mes", "diferencia"],
        "valor": [n_rows, n_unique_pozo_mes, n_rows - n_unique_pozo_mes],
    }
)

unit_check

,metrica,valor
0,filas,405996
1,combinaciones_unicas_idpozo_anio_mes,405996
2,diferencia,0


## 8. Tablas de resumen

Se consolidan tablas utiles para documentar el estado inicial de los datos.

### 8.1 Resumen de columnas

In [31]:
column_summary = pd.DataFrame(
    {
        "columna": df_base.columns,
        "tipo_dato": df_base.dtypes.astype(str).values,
        "nulos": df_base.isna().sum().values,
        "pct_nulos": (df_base.isna().mean() * 100).values,
        "valores_unicos": df_base.nunique(dropna=True).values,
    }
).sort_values("columna")

column_summary

,columna,tipo_dato,nulos,pct_nulos,valores_unicos
1,anio,int64,0,0.0000,21
27,areapermisoconcesion,str,0,0.0000,116
29,areayacimiento,str,0,0.0000,155
36,clasificacion,str,910,0.2241,3
32,coordenadax,float64,0,0.0000,4690
33,coordenaday,float64,0,0.0000,4435
30,cuenca,str,0,0.0000,4
21,empresa,str,0,0.0000,55
39,fecha_data,datetime64[us],0,0.0000,244
17,fechaingreso,datetime64[us],0,0.0000,9949


### 8.2 Resumen de nulos

In [32]:
null_summary

,nulos,pct_nulos,no_nulos,valores_unicos
vida_util,397379,97.8776,8617,466
observaciones,382947,94.3229,23049,215
clasificacion,910,0.2241,405086,3
subclasificacion,910,0.2241,405086,8
tipoextraccion,605,0.1490,405391,11
tipoestado,605,0.1490,405391,16
tipopozo,605,0.1490,405391,6
sub_tipo_recurso,440,0.1084,405556,2
idempresa,0,0.0000,405996,55
anio,0,0.0000,405996,21


### 8.3 Resumen de variables numericas

In [33]:
numeric_summary = df_base.select_dtypes(include=np.number).describe().T
numeric_summary

,count,mean,std,min,25%,50%,75%,max
anio,"405,996.0000","2,020.6801",4.0798,"2,006.0000","2,018.0000","2,022.0000","2,024.0000","2,026.0000"
mes,"405,996.0000",6.4448,3.4863,1.0000,3.0000,6.0000,10.0000,12.0000
idpozo,"405,996.0000","150,889.3071","18,382.6914","3,640.0000","146,227.0000","156,325.0000","160,105.0000","167,247.0000"
prod_pet,"405,996.0000",323.0807,919.7607,-0.0010,0.0000,7.5400,113.6025,"26,593.2610"
prod_gas,"405,996.0000",628.0281,"1,612.2393",-12.2670,8.1679,104.3605,452.7268,"29,129.6600"
prod_agua,"405,996.0000",182.2083,607.6422,0.0000,0.0000,12.4200,84.0112,"34,792.9500"
iny_agua,"405,996.0000",25.1099,863.1535,0.0000,0.0000,0.0000,0.0000,"70,278.8500"
iny_gas,"405,996.0000",36.5834,"8,736.6611",0.0000,0.0000,0.0000,0.0000,"3,569,000.0000"
iny_co2,"405,996.0000",0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
iny_otro,"405,996.0000",0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000


### 8.4 Resumen de variables categoricas

In [34]:
categorical_cols = df_base.select_dtypes(include=["object", "category", "bool"]).columns

categorical_summary = pd.DataFrame(
    [
        {
            "columna": col,
            "nulos": int(df_base[col].isna().sum()),
            "pct_nulos": float(df_base[col].isna().mean() * 100),
            "valores_unicos": int(df_base[col].nunique(dropna=True)),
            "valor_mas_frecuente": df_base[col].mode(dropna=True).iloc[0] if not df_base[col].mode(dropna=True).empty else np.nan,
            "frecuencia_valor_mas_frecuente": int(df_base[col].value_counts(dropna=True).iloc[0]) if not df_base[col].value_counts(dropna=True).empty else 0,
        }
        for col in categorical_cols
    ]
).sort_values("valores_unicos", ascending=False)

categorical_summary

,columna,nulos,pct_nulos,valores_unicos,valor_mas_frecuente,frecuencia_valor_mas_frecuente
8,sigla,0,0.0000,4744,YPF.Nq.RN.xp-58,708
4,observaciones,382947,94.3229,215,Cargado automáticamente como [Sin movimientos],7213
13,idareayacimiento,0,0.0000,156,LCLL,74053
14,areayacimiento,0,0.0000,155,LOMA CAMPANA-LLL,74053
11,idareapermisoconcesion,0,0.0000,116,LCAM,94482
12,areapermisoconcesion,0,0.0000,116,LOMA CAMPANA,94482
0,idempresa,0,0.0000,55,YPF,221496
7,empresa,0,0.0000,55,YPF S.A.,221496
10,formacion,0,0.0000,29,vaca muerta,206472
9,formprod,0,0.0000,29,VMUT,206472


### 8.5 Resumen temporal

In [35]:
temporal_summary = pd.DataFrame(
    {
        "metrica": [
            "fecha_min",
            "fecha_max",
            "meses_unicos",
            "anios_unicos",
            "pozos_unicos",
            "registros_minimos_por_pozo",
            "registros_medianos_por_pozo",
            "registros_maximos_por_pozo",
        ],
        "valor": [
            temporal_series.min(),
            temporal_series.max(),
            periodo_mes.nunique(dropna=True),
            df_base["anio"].nunique(dropna=True),
            df_base["idpozo"].nunique(dropna=True),
            records_by_well.min(),
            records_by_well.median(),
            records_by_well.max(),
        ],
    }
)

temporal_summary

,metrica,valor
0,fecha_min,2006-01-31 00:00:00
1,fecha_max,2026-04-30 00:00:00
2,meses_unicos,244
3,anios_unicos,21
4,pozos_unicos,4929
5,registros_minimos_por_pozo,1
6,registros_medianos_por_pozo,80.0000
7,registros_maximos_por_pozo,244


## 9. Guardado de version base procesada

Se guarda una version base solo con correcciones minimas: nombres de columnas normalizados, fechas convertidas y duplicados exactos eliminados solo si existieran. No se crea target, no se agregan features y no se unen otros datasets.

In [36]:
df_base_processed = df_base.copy()

if exact_duplicates_count > 0:
    df_base_processed = df_base_processed.drop_duplicates().copy()

processed_path = Path("../data/processed/produccion_no_convencional_base.csv")
processed_path.parent.mkdir(parents=True, exist_ok=True)
df_base_processed.to_csv(processed_path, index=False)

pd.DataFrame(
    {
        "metrica": [
            "filas_originales",
            "columnas_originales",
            "duplicados_exactos_eliminados",
            "filas_base_procesada",
            "columnas_base_procesada",
            "archivo_guardado",
        ],
        "valor": [
            df_raw.shape[0],
            df_raw.shape[1],
            exact_duplicates_count,
            df_base_processed.shape[0],
            df_base_processed.shape[1],
            str(processed_path),
        ],
    }
)

,metrica,valor
0,filas_originales,405996
1,columnas_originales,40
2,duplicados_exactos_eliminados,0
3,filas_base_procesada,405996
4,columnas_base_procesada,40
5,archivo_guardado,..\data\processed\produccion_no_convencional_b...


## 10. Resumen ejecutivo calculado

In [37]:
executive_summary = pd.DataFrame(
    {
        "metrica": [
            "dimensiones_dataset_original",
            "dimensiones_base_procesada",
            "rango_temporal",
            "cantidad_pozos_unicos",
            "duplicados_exactos",
            "duplicados_pozo_mes_grupos",
            "columnas_con_mas_50_pct_nulos",
            "valores_negativos_en_produccion",
            "archivo_base_procesada",
        ],
        "valor": [
            f"{df_raw.shape[0]:,} filas x {df_raw.shape[1]:,} columnas",
            f"{df_base_processed.shape[0]:,} filas x {df_base_processed.shape[1]:,} columnas",
            f"{temporal_series.min().date()} a {temporal_series.max().date()}",
            f"{df_base['idpozo'].nunique(dropna=True):,}",
            exact_duplicates_count,
            len(pozo_mes_duplicate_groups),
            ", ".join(high_null_columns.index.tolist()) if not high_null_columns.empty else "Ninguna",
            production_quality["negativos"].to_dict(),
            str(processed_path),
        ],
    }
)

executive_summary

,metrica,valor
0,dimensiones_dataset_original,"405,996 filas x 40 columnas"
1,dimensiones_base_procesada,"405,996 filas x 40 columnas"
2,rango_temporal,2006-01-31 a 2026-04-30
3,cantidad_pozos_unicos,"4,929"
4,duplicados_exactos,0
5,duplicados_pozo_mes_grupos,0
6,columnas_con_mas_50_pct_nulos,"vida_util, observaciones"
7,valores_negativos_en_produccion,"{'prod_pet': 1, 'prod_gas': 2, 'prod_agua': 0}"
8,archivo_base_procesada,..\data\processed\produccion_no_convencional_b...


## Conclusiones de calidad de datos

- **Aptitud para continuar:** el dataset es apto para continuar hacia EDA, con alertas de calidad que deben tratarse antes del modelado.
- **Unidad de analisis aparente:** la unidad parece ser pozo-mes. En la corrida actual, `idpozo + anio + mes` no presenta duplicados.
- **Columnas importantes:** `idpozo`, `anio`, `mes`, `fecha_data`, `prod_pet`, `prod_gas`, `prod_agua`, `empresa`, `provincia`, `cuenca`, `areayacimiento`, `areapermisoconcesion`, `tipoestado`, `tipoextraccion`, `tipopozo`, `tipo_de_recurso`, `sub_tipo_recurso`, `formacion`, `profundidad`, `coordenadax` y `coordenaday`.
- **Problemas de calidad detectados:** hay columnas con nulos muy altos (`vida_util` y `observaciones`), algunos nulos menores en variables descriptivas, pocos valores negativos en produccion y una cantidad relevante de producciones iguales a cero.
- **Duplicados por pozo-mes:** no se detectan grupos duplicados por `idpozo + anio + mes` en la corrida actual.
- **Valores nulos relevantes:** `vida_util` supera 97% de nulos y `observaciones` supera 94%. El resto de nulos relevantes parece bajo, pero debe revisarse en contexto.
- **Valores negativos o ceros en produccion:** aparecen pocos negativos en `prod_pet` y `prod_gas`; los ceros son frecuentes en `prod_pet`, `prod_gas` y `prod_agua`. No se corrigen automaticamente en este notebook.
- **Decisiones pendientes para EDA:** definir variable productiva principal, revisar negativos, interpretar ceros segun estado operativo, evaluar pozos con pocos meses de historia, tratar el periodo parcial 2026 y decidir como usar variables operativas sin leakage.
- **Construccion futura del target:** el dataset permite construir un target futuro de caida critica porque tiene identificador de pozo, fecha mensual y produccion historica. La definicion debe hacerse con `shift` temporal por pozo y con split temporal para evitar fuga de informacion.